In [ ]:

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM  # Punkt 4 benötigt
import pickle

# Load from a file
with open('../../Dataset/golden10K_data.pkl', 'rb') as file:
    df = pickle.load(file)

# ============================================
# 1. Chunking
# ============================================
chunks = []
metadaten = []  # parallel zu chunks speichern

for idx, row in df.iterrows():
    abstract = row['abstract']
    # Einfaches Chunking nach Sätzen
    for satz in abstract.split('. '):
        if satz:
            chunks.append(satz.strip())
            # Metadaten zu diesem Chunk speichern
            metadaten.append({
                'pmcid': row['pmcid'],
                'title': row['title'],
                'year': row['year'],
                'chunk_index': len([c for c in chunks if c.endswith(satz)])  # vereinfacht
            })

# ============================================
# 2. Embedding
# ============================================
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
chunk_embeddings = embed_model.encode(chunks)

# ============================================
# 3. Retrieval mit Metadaten
# ============================================
def retrieve_with_metadata(query, k=2):
    query_emb = embed_model.encode([query])[0]
    similarities = np.dot(chunk_embeddings, query_emb) / (
        np.linalg.norm(chunk_embeddings, axis=1) * np.linalg.norm(query_emb)
    )
    top_k_idx = np.argsort(similarities)[-k:][::-1]
    
    # Chunks UND Metadaten zurückgeben
    result_chunks = [chunks[i] for i in top_k_idx]
    result_metadata = [metadaten[i] for i in top_k_idx]
    return result_chunks, result_metadata

# ============================================
# 4. LLM laden (hier FLAN-T5-small)
# ============================================
print("Lade LLM...")
llm_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(llm_name)
llm = AutoModelForSeq2SeqLM.from_pretrained(llm_name)

# ============================================
# 5. Antwortgenerierung definieren
# ============================================
def generate_answer(query):
    # Relevante Chunks holen (Korrektur: Funktion heißt retrieve_with_metadata)
    gefundene, _ = retrieve_with_metadata(query, k=2)
    kontext = " ".join(gefundene)

    # Prompt im Stil von FLAN-T5 (instruction-tuned)
    prompt = f"Answer the question based on the given context. Only use: yes, no and maybe! \nContext: {kontext}\nQuestion: {query}\nAnswer:"

    # LLM aufrufen
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = llm.generate(
        **inputs, 
        max_new_tokens=100, 
        do_sample=True,        # WICHTIG: erst damit wird Temperatur aktiv
        temperature=0.3
    )
    antwort = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return antwort



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5809.99it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Lade LLM...


Loading weights: 100%|██████████| 190/190 [00:00<00:00, 3966.38it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
import time
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# ============================================================================
# 1. Modellaufbau: Chunking, Embedding, Retrieval, LLM
# ============================================================================
print("Lade Daten für Modell...")
with open('../../Dataset/golden10K_data.pkl', 'rb') as file:
    df = pickle.load(file)

# Chunking
chunks = []
metadaten = []
for idx, row in df.iterrows():
    abstract = row['abstract']
    for satz in abstract.split('. '):
        if satz:
            chunks.append(satz.strip())
            metadaten.append({
                'pmcid': row['pmcid'],          # interne PMCID für Retrieval
                'title': row['title'],
                'year': row['year'],
                'chunk_index': len([c for c in chunks if c.endswith(satz)])
            })

print(f"Erstellt {len(chunks)} Chunks.")

# Embedding
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
chunk_embeddings = embed_model.encode(chunks)

def retrieve_with_metadata(query, k=2):
    k = min(k, len(chunks))
    query_emb = embed_model.encode([query])[0]
    similarities = np.dot(chunk_embeddings, query_emb) / (
        np.linalg.norm(chunk_embeddings, axis=1) * np.linalg.norm(query_emb)
    )
    top_k_idx = np.argsort(similarities)[-k:][::-1]
    result_chunks = [chunks[i] for i in top_k_idx]
    result_metadata = [metadaten[i] for i in top_k_idx]
    return result_chunks, result_metadata

# LLM laden (FLAN-T5-large)
print("Lade LLM...")
llm_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(llm_name)
llm = AutoModelForSeq2SeqLM.from_pretrained(llm_name)

def generate_answer(query):
    gefundene, _ = retrieve_with_metadata(query, k=10)
    kontext = " ".join(gefundene)
    prompt = f"""Answer with exactly one word: 'yes', 'no', or 'maybe'.
Based on the given context, answer the question.
Context: {kontext}
Question: {query}
Answer:"""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = llm.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False,
        num_beams=1
    )
    antwort = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return antwort.strip()

# ============================================================================
# 1. Evaluation mit goldenen Fragen
# ============================================================================
def parse_answer(text: str) -> str:
    text = text.strip().lower()
    if "yes" in text:
        return "yes"
    if "no" in text:
        return "no"
    if "maybe" in text:
        return "maybe"
    return "unknown"

print("\nLade Fragen mit Ground Truth...")
df_questions = pd.read_pickle('../../Dataset/golden69_questions.pkl')

# Spalten prüfen
required_cols = ['question', 'final_decision']
if not all(col in df_questions.columns for col in required_cols):
    raise ValueError(f"Fehlende Spalten in goldenen Fragen: {required_cols}")

# Prüfen, ob die Spalte für die PMCID existiert (hier 'Accession ID')
if 'Accession ID' not in df_questions.columns:
    print("WARNUNG: Spalte 'Accession ID' nicht im Fragen-DataFrame. Retrieval-Metriken werden nicht berechnet.")
    has_retrieval_gt = False
else:
    has_retrieval_gt = True

true_labels = []
pred_labels = []
hit_rates = []
mrr_scores = []
latencies = []

for _, row in tqdm(df_questions.iterrows(), total=len(df_questions)):
    start = time.time()
    question = row['question']
    true = row['final_decision'].strip().lower()

    # Retrieval
    chunks_ret, meta_ret = retrieve_with_metadata(question, k=5)
    retrieved_pmcids = [m.get('pmcid') for m in meta_ret] if has_retrieval_gt else []

    # Retrieval-Metriken (nur wenn Ground Truth vorhanden)
    if has_retrieval_gt:
        true_accid = row.get('Accession ID', None)   # z.B. "PMC8005659"
        if true_accid is not None:
            hit = 1 if true_accid in retrieved_pmcids else 0
            hit_rates.append(hit)
            if hit:
                rank = retrieved_pmcids.index(true_accid) + 1
                mrr_scores.append(1/rank)
            else:
                mrr_scores.append(0)
        else:
            # Falls Zeile keine Accession ID hat
            hit_rates.append(0)
            mrr_scores.append(0)

    # Generation
    answer = generate_answer(question)
    pred = parse_answer(answer)

    true_labels.append(true)
    pred_labels.append(pred)

    latencies.append(time.time() - start)

# ============================================================================
# 2. Ausgabe der Metriken
# ============================================================================
valid_indices = [i for i, p in enumerate(pred_labels) if p in ['yes', 'no', 'maybe']]
true_valid = [true_labels[i] for i in valid_indices]
pred_valid = [pred_labels[i] for i in valid_indices]

accuracy = accuracy_score(true_valid, pred_valid)
labels = ['yes', 'no', 'maybe']
precision, recall, f1, _ = precision_recall_fscore_support(true_valid, pred_valid, labels=labels, average=None)
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(true_valid, pred_valid, average='macro')
cm = confusion_matrix(true_valid, pred_valid, labels=labels)

print("\n" + "="*60)
print("EVALUATIONSERGEBNISSE")
print("="*60)
print(f"Anzahl ausgewerteter Fragen: {len(df_questions)}")
print(f"Davon 'unknown'-Antworten (nicht parsebar): {pred_labels.count('unknown')}")
print(f"Gültige Vergleiche (yes/no/maybe): {len(true_valid)}")
print(f"\nAccuracy (über alle gültigen): {accuracy:.4f}")
print("\nKlassenweise Metriken:")
for i, label in enumerate(labels):
    print(f"  {label.upper()}  - Precision: {precision[i]:.4f}, Recall: {recall[i]:.4f}, F1: {f1[i]:.4f}")
print(f"\nMakro-Durchschnitt (ungewichtet): Precision: {macro_precision:.4f}, Recall: {macro_recall:.4f}, F1: {macro_f1:.4f}")
print("\nConfusion Matrix (Zeilen = wahr, Spalten = vorhergesagt):")
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
print(cm_df)

if has_retrieval_gt and hit_rates:
    print(f"\nRetrieval-Metriken (k=5):")
    print(f"  Hit Rate@5: {np.mean(hit_rates):.4f}")
    print(f"  Mean Reciprocal Rank (MRR): {np.mean(mrr_scores):.4f}")
else:
    print("\nRetrieval-Metriken wurden nicht berechnet (keine 'Accession ID' in goldenen Fragen).")

print(f"\nDurchschnittliche Latenz pro Frage: {np.mean(latencies):.2f} Sekunden")

df_results = pd.DataFrame({
    'question': df_questions['question'],
    'true_label': true_labels,
    'pred_label': pred_labels
})
df_results.to_csv('rag_evaluation_results.csv', index=False)
print("\nErgebnisse wurden in 'rag_evaluation_results.csv' gespeichert.")

Lade Daten für Modell...


KeyError: 'key of type tuple not found and not a MultiIndex'

In [4]:
frage = "What are risk factors for cancer?"
antwort = generate_answer(frage)
print("\nQuestion:", frage)
print("Answer:", antwort)


Question: What are risk factors for cancer?
Answer: environmental, genetic and epigenetic


In [5]:
frage = "What do genes do?"
antwort = generate_answer(frage)
print("\nQuestion:", frage)
print("Answer:", antwort)


Question: What do genes do?
Answer: regulate the expression of genes that are engaged in a variety of cellular physiological functions


In [10]:
frage = "What neurons comprise the leeches tactile system?"
antwort = generate_answer(frage)
print("\nQuestion:", frage)
print("Answer:", antwort)


Question: What neurons comprise the leeches tactile system?
Answer: thalamus


In [16]:
frage = "What is the meaning of sanguivorous?"
antwort = generate_answer(frage)
print("\nQuestion:", frage)
print("Answer:", antwort)


Question: What is the meaning of sanguivorous?
Answer: a fungus


In [18]:
frage = "What is biology"
antwort = generate_answer(frage)
print("\nQuestion:", frage)
print("Answer:", antwort)


Question: What is biology
Answer: an experimental discipline


In [9]:
df.shape

(939, 16)

In [8]:
# Load from a file
with open('../Dataset/1K_sample.pkl', 'rb') as file:
    df = pickle.load(file)

In [10]:
df.head()

,filename,pmcid,doi,journal,year,authors,title,keywords,abstract,introduction,methods,results,discussion,conclusion,tables,metadata_junk
0,PMC8005659.xml,PMC8005659,10.1016/j.bioactmat.2021.02.039,Bioactive Materials,2021,Zheng Kai; Bai Jiaxiang; Li Ning; Li Meng; Sun...,Protective effects of sirtuin 3 on titanium pa...,NLRP3 inflammasome; Osteoblast; Periprosthetic...,Periprosthetic osteolysis (PPO) remains the ke...,Total joint arthroplasties (TJAs) are common s...,All procedures for clinical sample collection ...,To investigate the efficacy of SIRT3 in wear p...,Periprosthetic osteolysis is an almost unavoid...,"Comprehensively, our study indicated that SIRT...",[],\n[credit authorship contribution statement]: ...
1,PMC8006602.xml,PMC8006602,10.1002/ehf2.13207,ESC Heart Failure,2021,Krzesiński Paweł; Siebert Janusz; Jankowska Ew...,Nurse‐led ambulatory care supported by non‐inv...,Congestion; Discharge; Haemodynamics; Heart fa...,Heart failure (HF) is characterized by frequen...,Heart failure (HF) is a complex clinical syndr...,"This study had a multicentre (three centres), ...","Of 73 recruited patients, three withdrew from ...",The results of this pilot study demonstrated t...,A 1 month period of care based on regular visi...,[**Alarm grading and recommendations**\n\n| Al...,\n[conflict of interest]: The authors declare ...
2,PMC8007390.xml,PMC8007390,10.5624/isd.20200144,Imaging Science in Dentistry,2021,Tolentino Elen de Souza; Amoroso-Silva Pablo A...,Comparison of limited- and large-volume cone-b...,Cone-Beam Computed Tomography; Endodontics; Mo...,This study was performed to compare the abilit...,"An isthmus, defined as narrow extension from e...",After receiving approval from the ethics commi...,"In the micro-CT images, the major thickness of...",Anatomical complexities such as isthmuses are ...,,[],
3,PMC8008312.xml,PMC8008312,10.2478/hukin-2021-0015,Journal of Human Kinetics,2021,Picabea Jon Mikel; Cámara Jesús; Nakamura Fabi...,Comparison of Heart Rate Variability before an...,autonomous nervous system; competition; fatigu...,The aim of this study was to compare heart rat...,Table tennis is a racquet sport that requires ...,Twenty-one table tennis players (21.86 ± 8.34 ...,Matches had average duration of 15.33 ± 3.89 m...,The purpose of this study was to compare the H...,,[**Heart rate variability (HRV) changes in the...,
4,PMC8009217.xml,PMC8009217,10.1136/gpsych-2020-100423,General Psychiatry,2021,Jiang Ping; Sun Xiujia; Ren Juanjuan; Liu Hong...,Effects of the combination of second-generatio...,Schizophrenia,Aripiprazole (ARI) is often prescribed alone o...,Aripiprazole (ARI) is a novel second-generatio...,To investigate the pharmacokinetic effects of ...,A total of 299 patients (106 men and 193 women...,"Schizophrenia is a complex, multidimensional d...",,[**Demographic data and data related to serum ...,
